In [1]:
# In this notebook I will prepare all data files

In [9]:
# AD based differential expression first
import pandas as pd
import os
#reran aug 11th

# Base DE files directory
base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/Differential_Expression_Final/Fixed"
files = [
    "poisson_DE_results_In.csv",
    "poisson_DE_results_Mic.csv",
    "poisson_DE_results_Oli.csv",
    "poisson_DE_results_Opc.csv",
    "poisson_DE_results_Ast.csv",
    "poisson_DE_results_Ex.csv"
]

# Map short codes to pretty names
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligodendrocyte Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}

# Output workbook path
out_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/AD_Differential_Expression.xlsx"

# Create an Excel writer
with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    # Add title page
    title_text = (
        "This workbook contains the full differential expression analysis results between Alzheimer's samples and controls."
        "Poisson model for Alzheimer's disease vs. Control across major brain cell types. "
        "Sheets correspond to individual cell types."
        "Bulk DEG analysis results are shown in the bulk sheet."
        "Directionality Comparison is also available."
    )
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="Title_Page", index=False)
    
    # Add each cell type sheet
    for file in files:
        cell_short = file.replace("poisson_DE_results_", "").replace(".csv", "")
        cell_name = ct_labels[cell_short]
        
        # Load data
        df = pd.read_csv(os.path.join(base_dir, file))

        #df["DEG"] = (df["p_adj"] < 0.05)
        df["DEG"] = (df["p_adj"] < 0.05) & (df["log2FC"].abs() > 0.25)
        
        # Write to sheet
        df.to_excel(writer, sheet_name=cell_name, index=False)

    # === Add Bulk sheet ===
    bulk_csv_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/AD_prediction_stuff_new/Bulk/zscore_directionality_matrix.csv"
    bulk_df = pd.read_csv(bulk_csv_path)
    bulk_df.to_excel(writer, sheet_name="Bulk", index=False)

    agree_csv_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/AD_prediction_stuff_new/Figure3/singlecell_vs_bulk_directionality_agreement_summary.csv"
    agree_df = pd.read_csv(bulk_csv_path)
    agree_df.to_excel(writer, sheet_name="SCvBulk", index=False)

print(f"Workbook saved to: {out_path}")

/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/AD_Differential_Expression.xlsx


In [8]:
# AD based differential expression first
import pandas as pd
import os
# reran aug 11th

# Base DE files directory
base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/Differential_Expression_Final/CERAD"
files = [
    "poisson_DE_results_In.csv",
    "poisson_DE_results_Mic.csv",
    "poisson_DE_results_Oli.csv",
    "poisson_DE_results_Opc.csv",
    "poisson_DE_results_Ast.csv",
    "poisson_DE_results_Ex.csv"
]

# Map short codes to pretty names
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligodendrocyte Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}

# Output workbook path
out_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/CERAD_Differential_Expression.xlsx"

# Create an Excel writer
with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    # Add title page
    title_text = (
        "This workbook contains the full differential expression analysis results between AD-like CERAD pathology samples and controls."
        "Poisson model for Alzheimer's disease vs. Control (based on CERAD pathology) across major brain cell types. "
        "Sheets correspond to individual cell types."
    )
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="CERAD_Title_Page", index=False)
    
    # Add each cell type sheet
    for file in files:
        cell_short = file.replace("poisson_DE_results_", "").replace(".csv", "")
        cell_name = ct_labels[cell_short]
        
        # Load data
        df = pd.read_csv(os.path.join(base_dir, file))

        # df["DEG"] = (df["p_adj"] < 0.05)
        df["DEG"] = (df["p_adj"] < 0.05) & (df["log2FC"].abs() > 0.25)
        
        # Write to sheet
        df.to_excel(writer, sheet_name=cell_name, index=False)

print(f"Workbook saved to: {out_path}")

/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/CERAD_Differential_Expression.xlsx


In [3]:
# Updated as of July4th
import os
import pandas as pd
import numpy as np
import joblib
from collections import defaultdict
from sklearn.metrics import roc_curve, auc
# reran aug 11th

# === Paths ===
base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
out_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/AD_GenesModel_Feature_Importances_CellLevel.xlsx"

# === Cell type labels ===
## FIX: Shortened "Oligodendrocyte Progenitor Cells" to prevent Excel repair errors.
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligo Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}
cell_types = list(ct_labels.keys())

# === Description for title page ===
title_text = (
    "This workbook shows feature importances and ROC AUC values for gene expression based classifiers predicting on AD.\n"
    "This workbook contains two main tables:\n"
    "Left ('Filtered'): Shows feature importances normalized against other non-zero features within each split. Zeros are shown as blank (NaN).\n"
    "Right ('Raw'): Shows the raw, unnormalized feature importances from the model."
)

# === Create Excel writer ===
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    # Title page
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="Title_Page", index=False)

    overview_rows = []

    for subcluster, label in ct_labels.items():
        aucs = []
        row = {"Cell Type": label}
        for i in range(1, 6):
            pred_file = os.path.join(base_dir, subcluster, f"split_{i}", "test_predictions.csv")
            if not os.path.exists(pred_file):
                auc_value = np.nan
            else:
                df = pd.read_csv(pred_file)
                y_true = df["true_label"]
                y_score = df["predicted_proba"]
                fpr, tpr, _ = roc_curve(y_true, y_score)
                auc_value = auc(fpr, tpr)
            aucs.append(auc_value)
            row[f"Split_{i}"] = auc_value
        row["Mean"] = np.nanmean(aucs)
        row["Std"] = np.nanstd(aucs)
        overview_rows.append(row)
    overview_df = pd.DataFrame(overview_rows)
    overview_df.to_excel(writer, sheet_name="Overview_AUCs", index=False, float_format="%.3f")

    for cell_type in cell_types:
        print(f"Processing: {cell_type}")

        ## REFACTORED: Load all data once to ensure consistency and efficiency.
        all_split_data = {}
        all_features = set()

        for split in range(1, 6):
            joblib_path = os.path.join(base_dir, cell_type, f"split_{split}", "maximal_classifier.joblib")
            if not os.path.exists(joblib_path):
                all_split_data[split] = {} # Handle missing splits
                continue

            model = joblib.load(joblib_path)
            feature_names = model.feature_names_in_
            importances = model.feature_importances_

            all_features.update(feature_names)
            all_split_data[split] = dict(zip(feature_names, importances))

        # --- Create Filtered and Raw DataFrames from the single source of data ---
        table_rows = []
        sorted_features = sorted(list(all_features))

        for feature in sorted_features:
            raw_values = []
            filtered_norm_values = []

            # Correctly map values for each split
            for split in range(1, 6):
                # Get raw importance for this feature in this split
                raw_imp = all_split_data.get(split, {}).get(feature, 0.0)
                raw_values.append(raw_imp)

                # Get all importances for this split to calculate the total
                all_imps_in_split = np.array(list(all_split_data.get(split, {}).values()))
                
                ## FIX: Use np.nansum for robust handling of potential NaN values in data.
                total_nonzero_imp = np.nansum(all_imps_in_split[all_imps_in_split > 0])

                # Calculate filtered normalized value
                if raw_imp > 0 and total_nonzero_imp > 0:
                    norm_imp = raw_imp / total_nonzero_imp
                else:
                    norm_imp = 0.0

                filtered_norm_values.append(norm_imp)

            # Calculate stats
            filtered_mean = np.mean(filtered_norm_values)
            filtered_std = np.std(filtered_norm_values)
            
            raw_mean = np.mean(raw_values)
            raw_std = np.std(raw_values)
            
            # Combine all data for the final row
            row_data = [feature] + filtered_norm_values + [filtered_mean, filtered_std] + raw_values + [raw_mean, raw_std]
            table_rows.append(row_data)

        # Define columns for the final merged table
        filtered_cols = [f"Filtered_Norm_Split_{i}" for i in range(1, 6)] + ["Filtered_Mean", "Filtered_Std"]
        raw_cols = [f"Raw_Imp_Split_{i}" for i in range(1, 6)] + ["Raw_Mean", "Raw_Std"]
        
        final_columns = ["Feature"] + filtered_cols + raw_cols
        
        # Create the final DataFrame
        final_df = pd.DataFrame(table_rows, columns=final_columns)

        # Insert a blank column to separate the two tables
        final_df.insert(len(filtered_cols) + 1, "   ", "")

        # Write to Excel
        sheet_name = ct_labels[cell_type]
        final_df.to_excel(writer, sheet_name=sheet_name, index=False, float_format="%.6f")

print(f"Workbook saved to: {out_path}")

Processing: Ast
Processing: Mic
Processing: In
Processing: Oli
Processing: Opc
Processing: Ex
Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/AD_GenesModel_Feature_Importances_CellLevel.xlsx


In [5]:
import os
import pandas as pd
import numpy as np
import joblib
from collections import defaultdict

# === Paths ===
base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_both_new"
out_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/AD_CombinedModel_Feature_Importances_CellLevel.xlsx"

# === Cell type labels ===
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligodendrocyte Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}
cell_types = list(ct_labels.keys())


# === Description for title page ===
title_text = (
    "This workbook shows feature importances and ROC AUC values for combined gene expression, demographics and APOE genotype based classifiers predicting on AD.\n"
    "This workbook contains two main tables:\n"
    "Left ('Filtered'): Shows feature importances normalized against other non-zero features within each split. Zeros are shown as blank (NaN).\n"
    "Right ('Raw'): Shows the raw, unnormalized feature importances from the model."
)

# === Create Excel writer ===
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    # Title page
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="Title_Page", index=False)

    overview_rows = []

    for subcluster, label in ct_labels.items():
        aucs = []
        row = {"Cell Type": label}
        for i in range(1, 6):
            pred_file = os.path.join(base_dir, subcluster, f"split_{i}", "test_predictions.csv")
            if not os.path.exists(pred_file):
                auc_value = np.nan
            else:
                df = pd.read_csv(pred_file)
                y_true = df["true_label"]
                y_score = df["predicted_proba"]
                fpr, tpr, _ = roc_curve(y_true, y_score)
                auc_value = auc(fpr, tpr)
            aucs.append(auc_value)
            row[f"Split_{i}"] = auc_value
        row["Mean"] = np.nanmean(aucs)
        row["Std"] = np.nanstd(aucs)
        overview_rows.append(row)
    overview_df = pd.DataFrame(overview_rows)
    overview_df.to_excel(writer, sheet_name="Overview_AUCs", index=False, float_format="%.3f")

    for cell_type in cell_types:
        print(f"Processing: {cell_type}")

        ## REFACTORED: Load all data once to ensure consistency and efficiency.
        all_split_data = {}
        all_features = set()

        for split in range(1, 6):
            joblib_path = os.path.join(base_dir, cell_type, f"split_{split}", "maximal_classifier.joblib")
            if not os.path.exists(joblib_path):
                all_split_data[split] = {} # Handle missing splits
                continue

            model = joblib.load(joblib_path)
            feature_names = model.feature_names_in_
            importances = model.feature_importances_

            all_features.update(feature_names)
            all_split_data[split] = dict(zip(feature_names, importances))

        # --- Create Filtered and Raw DataFrames from the single source of data ---
        table_rows = []
        sorted_features = sorted(list(all_features))

        for feature in sorted_features:
            raw_values = []
            filtered_norm_values = []

            # Correctly map values for each split
            for split in range(1, 6):
                # Get raw importance for this feature in this split
                raw_imp = all_split_data.get(split, {}).get(feature, 0.0)
                raw_values.append(raw_imp)

                # Get all importances for this split to calculate the total
                all_imps_in_split = np.array(list(all_split_data.get(split, {}).values()))
                
                ## FIX: Use np.nansum for robust handling of potential NaN values in data.
                total_nonzero_imp = np.nansum(all_imps_in_split[all_imps_in_split > 0])

                # Calculate filtered normalized value
                if raw_imp > 0 and total_nonzero_imp > 0:
                    norm_imp = raw_imp / total_nonzero_imp
                else:
                    norm_imp = 0.0

                filtered_norm_values.append(norm_imp)

            # Calculate stats
            filtered_mean = np.mean(filtered_norm_values)
            filtered_std = np.std(filtered_norm_values)
            
            raw_mean = np.mean(raw_values)
            raw_std = np.std(raw_values)
            
            # Combine all data for the final row
            row_data = [feature] + filtered_norm_values + [filtered_mean, filtered_std] + raw_values + [raw_mean, raw_std]
            table_rows.append(row_data)

        # Define columns for the final merged table
        filtered_cols = [f"Filtered_Norm_Split_{i}" for i in range(1, 6)] + ["Filtered_Mean", "Filtered_Std"]
        raw_cols = [f"Raw_Imp_Split_{i}" for i in range(1, 6)] + ["Raw_Mean", "Raw_Std"]
        
        final_columns = ["Feature"] + filtered_cols + raw_cols
        
        # Create the final DataFrame
        final_df = pd.DataFrame(table_rows, columns=final_columns)

        # Insert a blank column to separate the two tables
        final_df.insert(len(filtered_cols) + 1, "   ", "")

        # Write to Excel
        sheet_name = ct_labels[cell_type]
        final_df.to_excel(writer, sheet_name=sheet_name, index=False, float_format="%.6f")

print(f"Workbook saved to: {out_path}")

Processing: Ast
Processing: Mic
Processing: In
Processing: Oli
Processing: Opc


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/openpyxl/workbook/child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Processing: Ex
Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/AD_CombinedModel_Feature_Importances_CellLevel.xlsx


In [6]:
import os
import pandas as pd
import numpy as np
import joblib
from collections import defaultdict

# === Paths ===
base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_genes_subcluster_new"
out_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/AD_GenesModel_Feature_Importances_SubclusterLevel.xlsx"

# === Cell type labels ===
## FIX: Shortened "Oligodendrocyte Progenitor Cells" to prevent Excel repair errors.
ct_labels = {
    "Ast0": "Astrocytes 0",
    "Ast1": "Astrocytes 1",
    "Ast2": "Astrocytes 2",
    "Ast3": "Astrocytes 3",

    "Ex0": "Excitatory 0",
    "Ex1": "Excitatory 1",
    "Ex2": "Excitatory 2",
    "Ex3": "Excitatory 3",
    "Ex4": "Excitatory 4",
    "Ex5": "Excitatory 5",
    "Ex6": "Excitatory 6",
    "Ex7": "Excitatory 7",
    "Ex8": "Excitatory 8",
    "Ex9": "Excitatory 9",
    "Ex11": "Excitatory 11",
    "Ex12": "Excitatory 12",
    "Ex14": "Excitatory 14",

    "In0": "Inhibitory 0",
    "In1": "Inhibitory 1",
    "In2": "Inhibitory 2",
    "In3": "Inhibitory 3",
    "In4": "Inhibitory 4",
    "In5": "Inhibitory 5",
    "In6": "Inhibitory 6",
    "In7": "Inhibitory 7",
    "In8": "Inhibitory 8",
    "In9": "Inhibitory 9",
    "In10": "Inhibitory 10",
    "In11": "Inhibitory 11",

    "Mic0": "Microglia 0",
    "Mic1": "Microglia 1",
    "Mic2": "Microglia 2",
    "Mic3": "Microglia 3",

    "Oli0": "Oligodendrocytes 0",
    "Oli1": "Oligodendrocytes 1",
    "Oli3": "Oligodendrocytes 3",
    "Oli4": "Oligodendrocytes 4",
    "Oli5": "Oligodendrocytes 5",

    "Opc0": "OPCs 0",
    "Opc1": "OPCs 1",
    "Opc2": "OPCs 2",
}

cell_types = list(ct_labels.keys())

# === Description for title page ===
title_text = (
    "This workbook shows feature importances and ROC AUC values for gene expression based classifiers predicting on AD.\n"
    "This workbook contains two main tables:\n"
    "Left ('Filtered'): Shows feature importances normalized against other non-zero features within each split. Zeros are shown as blank (NaN).\n"
    "Right ('Raw'): Shows the raw, unnormalized feature importances from the model."
)

# === Create Excel writer ===
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    # Title page
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="Title_Page", index=False)

    overview_rows = []

    for subcluster, label in ct_labels.items():
        aucs = []
        row = {"Cell Type": label}
        for i in range(1, 6):
            pred_file = os.path.join(base_dir, subcluster, f"split_{i}", "test_predictions.csv")
            if not os.path.exists(pred_file):
                auc_value = np.nan
            else:
                df = pd.read_csv(pred_file)
                y_true = df["true_label"]
                y_score = df["predicted_proba"]
                fpr, tpr, _ = roc_curve(y_true, y_score)
                auc_value = auc(fpr, tpr)
            aucs.append(auc_value)
            row[f"Split_{i}"] = auc_value
        row["Mean"] = np.nanmean(aucs)
        row["Std"] = np.nanstd(aucs)
        overview_rows.append(row)
    overview_df = pd.DataFrame(overview_rows)
    overview_df.to_excel(writer, sheet_name="Overview_AUCs", index=False, float_format="%.3f")

    for cell_type in cell_types:
        print(f"Processing: {cell_type}")

        ## REFACTORED: Load all data once to ensure consistency and efficiency.
        all_split_data = {}
        all_features = set()

        for split in range(1, 6):
            joblib_path = os.path.join(base_dir, cell_type, f"split_{split}", "maximal_classifier.joblib")
            if not os.path.exists(joblib_path):
                all_split_data[split] = {} # Handle missing splits
                continue

            model = joblib.load(joblib_path)
            feature_names = model.feature_names_in_
            importances = model.feature_importances_

            all_features.update(feature_names)
            all_split_data[split] = dict(zip(feature_names, importances))

        # --- Create Filtered and Raw DataFrames from the single source of data ---
        table_rows = []
        sorted_features = sorted(list(all_features))

        for feature in sorted_features:
            raw_values = []
            filtered_norm_values = []

            # Correctly map values for each split
            for split in range(1, 6):
                # Get raw importance for this feature in this split
                raw_imp = all_split_data.get(split, {}).get(feature, 0.0)
                raw_values.append(raw_imp)

                # Get all importances for this split to calculate the total
                all_imps_in_split = np.array(list(all_split_data.get(split, {}).values()))
                
                ## FIX: Use np.nansum for robust handling of potential NaN values in data.
                total_nonzero_imp = np.nansum(all_imps_in_split[all_imps_in_split > 0])

                # Calculate filtered normalized value
                if raw_imp > 0 and total_nonzero_imp > 0:
                    norm_imp = raw_imp / total_nonzero_imp
                else:
                    norm_imp = 0.0

                filtered_norm_values.append(norm_imp)

            # Calculate stats
            filtered_mean = np.mean(filtered_norm_values)
            filtered_std = np.std(filtered_norm_values)
            
            raw_mean = np.mean(raw_values)
            raw_std = np.std(raw_values)
            
            # Combine all data for the final row
            row_data = [feature] + filtered_norm_values + [filtered_mean, filtered_std] + raw_values + [raw_mean, raw_std]
            table_rows.append(row_data)

        # Define columns for the final merged table
        filtered_cols = [f"Filtered_Norm_Split_{i}" for i in range(1, 6)] + ["Filtered_Mean", "Filtered_Std"]
        raw_cols = [f"Raw_Imp_Split_{i}" for i in range(1, 6)] + ["Raw_Mean", "Raw_Std"]
        
        final_columns = ["Feature"] + filtered_cols + raw_cols
        
        # Create the final DataFrame
        final_df = pd.DataFrame(table_rows, columns=final_columns)

        # Insert a blank column to separate the two tables
        final_df.insert(len(filtered_cols) + 1, "   ", "")

        # Write to Excel
        sheet_name = ct_labels[cell_type]
        final_df.to_excel(writer, sheet_name=sheet_name, index=False, float_format="%.6f")

print(f"Workbook saved to: {out_path}")

/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/metrics/_ranking.py:1188: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/metrics/_ranking.py:1188: UndefinedMetricWarning: No positive samples in y_true, true positive value should be meaningless
  warnings.warn(


Processing: Ast0
Processing: Ast1
Processing: Ast2
Processing: Ast3
Processing: Ex0
Processing: Ex1
Processing: Ex2
Processing: Ex3
Processing: Ex4
Processing: Ex5
Processing: Ex6
Processing: Ex7
Processing: Ex8
Processing: Ex9
Processing: Ex11
Processing: Ex12
Processing: Ex14
Processing: In0
Processing: In1
Processing: In2
Processing: In3
Processing: In4
Processing: In5
Processing: In6
Processing: In7
Processing: In8
Processing: In9
Processing: In10
Processing: In11
Processing: Mic0
Processing: Mic1
Processing: Mic2
Processing: Mic3
Processing: Oli0
Processing: Oli1
Processing: Oli3
Processing: Oli4
Processing: Oli5
Processing: Opc0
Processing: Opc1
Processing: Opc2
Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/AD_GenesModel_Feature_Importances_SubclusterLevel.xlsx


In [7]:
import os
import pandas as pd
import numpy as np
import joblib
from collections import defaultdict

# === Paths ===
base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_both_cerad"
out_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/CERAD_CombinedModel_Feature_Importances_CellLevel.xlsx"

# === Cell type labels ===
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligodendrocyte Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}
cell_types = list(ct_labels.keys())

# === Description for title page ===
title_text = (
    "This workbook shows feature importances and ROC AUC values for the gene expression, demographics, and APOE genotype based classifiers predicting on CERAD pathology. \n"
    "This workbook summarizes feature importances using two approaches:\n"
    "Left: These columns include only non-zero importances, computing mean/std only across non-zero splits (as in the plot).\n"
    "Right: Raw data columns include zeros and compute mean/std across all splits.\n"
)

# === Create Excel writer ===
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    # Title page
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="Title_Page", index=False)

    overview_rows = []

    for subcluster, label in ct_labels.items():
        aucs = []
        row = {"Cell Type": label}
        for i in range(1, 6):
            pred_file = os.path.join(base_dir, subcluster, f"split_{i}", "test_predictions.csv")
            if not os.path.exists(pred_file):
                auc_value = np.nan
            else:
                df = pd.read_csv(pred_file)
                y_true = df["true_label"]
                y_score = df["predicted_proba"]
                fpr, tpr, _ = roc_curve(y_true, y_score)
                auc_value = auc(fpr, tpr)
            aucs.append(auc_value)
            row[f"Split_{i}"] = auc_value
        row["Mean"] = np.nanmean(aucs)
        row["Std"] = np.nanstd(aucs)
        overview_rows.append(row)
    overview_df = pd.DataFrame(overview_rows)
    overview_df.to_excel(writer, sheet_name="Overview_AUCs", index=False, float_format="%.3f")

    for cell_type in cell_types:
        print(f"Processing: {cell_type}")

        ## REFACTORED: Load all data once to ensure consistency and efficiency.
        all_split_data = {}
        all_features = set()

        for split in range(1, 6):
            joblib_path = os.path.join(base_dir, cell_type, f"split_{split}", "maximal_classifier.joblib")
            if not os.path.exists(joblib_path):
                all_split_data[split] = {} # Handle missing splits
                continue

            model = joblib.load(joblib_path)
            feature_names = model.feature_names_in_
            importances = model.feature_importances_

            all_features.update(feature_names)
            all_split_data[split] = dict(zip(feature_names, importances))

        # --- Create Filtered and Raw DataFrames from the single source of data ---
        table_rows = []
        sorted_features = sorted(list(all_features))

        for feature in sorted_features:
            raw_values = []
            filtered_norm_values = []

            # Correctly map values for each split
            for split in range(1, 6):
                # Get raw importance for this feature in this split
                raw_imp = all_split_data.get(split, {}).get(feature, 0.0)
                raw_values.append(raw_imp)

                # Get all importances for this split to calculate the total
                all_imps_in_split = np.array(list(all_split_data.get(split, {}).values()))
                
                ## FIX: Use np.nansum for robust handling of potential NaN values in data.
                total_nonzero_imp = np.nansum(all_imps_in_split[all_imps_in_split > 0])

                # Calculate filtered normalized value
                if raw_imp > 0 and total_nonzero_imp > 0:
                    norm_imp = raw_imp / total_nonzero_imp
                else:
                    norm_imp = 0.0

                filtered_norm_values.append(norm_imp)

            # Calculate stats
            filtered_mean = np.mean(filtered_norm_values)
            filtered_std = np.std(filtered_norm_values)
            
            raw_mean = np.mean(raw_values)
            raw_std = np.std(raw_values)
            
            # Combine all data for the final row
            row_data = [feature] + filtered_norm_values + [filtered_mean, filtered_std] + raw_values + [raw_mean, raw_std]
            table_rows.append(row_data)

        # Define columns for the final merged table
        filtered_cols = [f"Filtered_Norm_Split_{i}" for i in range(1, 6)] + ["Filtered_Mean", "Filtered_Std"]
        raw_cols = [f"Raw_Imp_Split_{i}" for i in range(1, 6)] + ["Raw_Mean", "Raw_Std"]
        
        final_columns = ["Feature"] + filtered_cols + raw_cols
        
        # Create the final DataFrame
        final_df = pd.DataFrame(table_rows, columns=final_columns)

        # Insert a blank column to separate the two tables
        final_df.insert(len(filtered_cols) + 1, "   ", "")

        # Write to Excel
        sheet_name = ct_labels[cell_type]
        final_df.to_excel(writer, sheet_name=sheet_name, index=False, float_format="%.6f")

print(f"Workbook saved to: {out_path}")


Processing: Ast


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 w

Processing: Mic


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 w

Processing: In


...

Processing: Opc


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 w

Processing: Ex


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 w

Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/CERAD_CombinedModel_Feature_Importances_CellLevel.xlsx


In [8]:
import os
import pandas as pd
import numpy as np
import joblib
from collections import defaultdict

# === Paths ===
base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_cerad"
out_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/CERAD_GenesModel_Feature_Importances_CellLevel.xlsx"

# === Cell type labels ===
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligodendrocyte Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}
cell_types = list(ct_labels.keys())

# === Description for title page ===
title_text = (
    "This workbook shows feature imporatances and ROC AUC values for gene expression based classifiers predicting on CERAD pathology. \n"
    "This workbook summarizes feature importances using two approaches:\n"
    "Left: These columns include only non-zero importances, computing mean/std only across non-zero splits (as in the plot).\n"
    "Right: Raw data columns include zeros and compute mean/std across all splits.\n"
)

# === Create Excel writer ===
with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    # Title page
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="Title_Page", index=False)

    overview_rows = []

    for subcluster, label in ct_labels.items():
        aucs = []
        row = {"Cell Type": label}
        for i in range(1, 6):
            pred_file = os.path.join(base_dir, subcluster, f"split_{i}", "test_predictions.csv")
            if not os.path.exists(pred_file):
                auc_value = np.nan
            else:
                df = pd.read_csv(pred_file)
                y_true = df["true_label"]
                y_score = df["predicted_proba"]
                fpr, tpr, _ = roc_curve(y_true, y_score)
                auc_value = auc(fpr, tpr)
            aucs.append(auc_value)
            row[f"Split_{i}"] = auc_value
        row["Mean"] = np.nanmean(aucs)
        row["Std"] = np.nanstd(aucs)
        overview_rows.append(row)
    overview_df = pd.DataFrame(overview_rows)
    overview_df.to_excel(writer, sheet_name="Overview_AUCs", index=False, float_format="%.3f")

    for cell_type in cell_types:
        print(f"Processing: {cell_type}")

        ## REFACTORED: Load all data once to ensure consistency and efficiency.
        all_split_data = {}
        all_features = set()

        for split in range(1, 6):
            joblib_path = os.path.join(base_dir, cell_type, f"split_{split}", "maximal_classifier.joblib")
            if not os.path.exists(joblib_path):
                all_split_data[split] = {} # Handle missing splits
                continue

            model = joblib.load(joblib_path)
            feature_names = model.feature_names_in_
            importances = model.feature_importances_

            all_features.update(feature_names)
            all_split_data[split] = dict(zip(feature_names, importances))

        # --- Create Filtered and Raw DataFrames from the single source of data ---
        table_rows = []
        sorted_features = sorted(list(all_features))

        for feature in sorted_features:
            raw_values = []
            filtered_norm_values = []

            # Correctly map values for each split
            for split in range(1, 6):
                # Get raw importance for this feature in this split
                raw_imp = all_split_data.get(split, {}).get(feature, 0.0)
                raw_values.append(raw_imp)

                # Get all importances for this split to calculate the total
                all_imps_in_split = np.array(list(all_split_data.get(split, {}).values()))
                
                ## FIX: Use np.nansum for robust handling of potential NaN values in data.
                total_nonzero_imp = np.nansum(all_imps_in_split[all_imps_in_split > 0])

                # Calculate filtered normalized value
                if raw_imp > 0 and total_nonzero_imp > 0:
                    norm_imp = raw_imp / total_nonzero_imp
                else:
                    norm_imp = 0.0

                filtered_norm_values.append(norm_imp)

            # Calculate stats
            filtered_mean = np.mean(filtered_norm_values)
            filtered_std = np.std(filtered_norm_values)
            
            raw_mean = np.mean(raw_values)
            raw_std = np.std(raw_values)
            
            # Combine all data for the final row
            row_data = [feature] + filtered_norm_values + [filtered_mean, filtered_std] + raw_values + [raw_mean, raw_std]
            table_rows.append(row_data)

        # Define columns for the final merged table
        filtered_cols = [f"Filtered_Norm_Split_{i}" for i in range(1, 6)] + ["Filtered_Mean", "Filtered_Std"]
        raw_cols = [f"Raw_Imp_Split_{i}" for i in range(1, 6)] + ["Raw_Mean", "Raw_Std"]
        
        final_columns = ["Feature"] + filtered_cols + raw_cols
        
        # Create the final DataFrame
        final_df = pd.DataFrame(table_rows, columns=final_columns)

        # Insert a blank column to separate the two tables
        final_df.insert(len(filtered_cols) + 1, "   ", "")

        # Write to Excel
        sheet_name = ct_labels[cell_type]
        final_df.to_excel(writer, sheet_name=sheet_name, index=False, float_format="%.6f")

print(f"Workbook saved to: {out_path}")

Processing: Ast


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 w

Processing: Mic


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 w

Processing: In


...

Processing: Opc


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 w

Processing: Ex


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 w

Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/CERAD_GenesModel_Feature_Importances_CellLevel.xlsx


In [4]:
# reran
import os
import pandas as pd
import joblib
from collections import defaultdict

# === Paths ===
base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
gwas_path = '/n/scratch/users/a/adm808/MONDO_0004975_associations_export.tsv'
output_path = '/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/AD_Predictive_Genes_GWAS_Filtered.csv'

# === Cell type labels ===
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligodendrocyte Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}
cell_types = list(ct_labels.keys())

# === Build predictive genes dictionary ===
predictive_genes_dict = {}

for cell_type in cell_types:
    print(f"Processing: {cell_type}")
    gene_presence = defaultdict(int)

    for split in range(1, 6):
        joblib_path = os.path.join(base_dir, cell_type, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(joblib_path):
            continue

        model = joblib.load(joblib_path)
        feature_names = model.feature_names_in_
        importances = model.feature_importances_

        for gene, imp in zip(feature_names, importances):
            if imp > 0:
                gene_presence[gene.upper()] += 1

    final_predictors = [gene for gene, count in gene_presence.items() if count >= 2]
    predictive_genes_dict[cell_type] = final_predictors
    print(f"{cell_type}: {len(final_predictors)} predictive genes")

# === Load GWAS dataset ===
gwas_df = pd.read_csv(gwas_path, sep='\t', low_memory=False)

# === Prepare all rows for export ===
all_rows = []

for cell_type, pred_genes in predictive_genes_dict.items():
    pred_genes_upper = set(g.upper() for g in pred_genes)

    # Get all GWAS genes
    all_gwas_genes = set()
    for genes_str in gwas_df['mappedGenes'].dropna():
        for g in genes_str.split(','):
            all_gwas_genes.add(g.strip().upper())

    # Manual exception: include RASGEF1B logic
    if "RASGEF1B" in pred_genes_upper and "RASGEF1B" in all_gwas_genes:
        pred_genes_upper.add("RASGEF1B")

    for idx, row in gwas_df.iterrows():
        mapped_genes_str = row['mappedGenes']
        if pd.isna(mapped_genes_str):
            continue

        genes_in_row = [g.strip().upper() for g in mapped_genes_str.split(',') if g.strip()]
        overlap_genes = pred_genes_upper & set(genes_in_row)

        for gene in overlap_genes:
            new_row = row.copy()

            # Special case handling for RASGEF1
            if gene == "RASGEF1B" and "RASGEF1B" in pred_genes_upper:
                predictor_gene_label = "RASGEF1B"
            else:
                predictor_gene_label = gene

            new_row['CellType'] = ct_labels[cell_type]
            new_row['Predictor_Gene'] = predictor_gene_label
            new_row['Predictor_Status'] = 'Predictive'
            all_rows.append(new_row)

# === Create final DataFrame ===
final_df = pd.DataFrame(all_rows)

# === Save ===
final_df.to_csv(output_path, index=False)
print(f"Saved filtered GWAS file with overlaps to: {output_path}")

Processing: Ast
Ast: 175 predictive genes
Processing: Mic
Mic: 466 predictive genes
Processing: In
In: 108 predictive genes
Processing: Oli
Oli: 621 predictive genes
Processing: Opc
Opc: 835 predictive genes
Processing: Ex
Ex: 162 predictive genes
Saved filtered GWAS file with overlaps to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/AD_Predictive_Genes_GWAS_Filtered.csv


In [ ]:
# reran
import os
import joblib
from collections import defaultdict

# Define base path and cell types
base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligodendrocyte Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}
cell_types = list(ct_labels.keys())

# Dictionary to hold final predictors
predictive_genes_dict = {}

for cell_type in cell_types:
    print(f"Processing: {cell_type}")
    gene_presence = defaultdict(int)

    for split in range(1, 6):
        joblib_path = os.path.join(base_dir, cell_type, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(joblib_path):
            continue

        model = joblib.load(joblib_path)
        feature_names = model.feature_names_in_
        importances = model.feature_importances_

        for gene, imp in zip(feature_names, importances):
            if imp > 0:
                gene_presence[gene] += 1

    # Keep genes that are non-zero in ≥ 3 splits
    final_predictors = [gene for gene, count in gene_presence.items() if count >= 2]
    predictive_genes_dict[cell_type] = final_predictors
    print(f"{cell_type}: {len(final_predictors)} predictive genes")

# This dictionary can now be used as your new top_25_dict equivalent
top_25_features_dict = predictive_genes_dict

import pandas as pd

# --- Load GWAS genes ---
gwas_path = '/n/scratch/users/a/adm808/MONDO_0004975_associations_export.tsv'
gwas_df = pd.read_csv(gwas_path, sep='\t', low_memory=False)

# Extract GWAS genes as uppercase
gwas_genes = set()
for genes in gwas_df['mappedGenes'].dropna():
    for g in genes.split(','):
        gwas_genes.add(g.strip().upper())

# --- Check overlap per cell type ---
gwas_overlap_dict = {}

for cell_type, pred_genes in predictive_genes_dict.items():
    pred_genes_upper = set(g.upper() for g in pred_genes)
    
    # Exact overlap
    overlap = pred_genes_upper & gwas_genes

    # Manual exception: include RASGEF1B if RASGEF1B in GWAS
    if "RASGEF1B" in pred_genes_upper and "RASGEF1B" in gwas_genes:
        overlap.add("RASGEF1B")

    gwas_overlap_dict[cell_type] = sorted(list(overlap))
    print(f"{cell_type}: {len(overlap)} genes overlap with GWAS (exact match + RASGEF1B manual inclusion)")
    print(overlap)

# --- Save CSV for supplement ---
all_types = []
for cell_type, overlap_genes in gwas_overlap_dict.items():
    for gene in overlap_genes:
        all_types.append({'CellType': cell_type, 'Gene': gene})

pd.DataFrame(all_types).to_csv('/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/check.csv', index=False)


Processing: Ast
Ast: 175 predictive genes
Processing: Mic
Mic: 466 predictive genes
Processing: In
In: 108 predictive genes
Processing: Oli
Oli: 621 predictive genes
Processing: Opc
Opc: 835 predictive genes
Processing: Ex
Ex: 162 predictive genes
Ast: 27 genes overlap with GWAS (exact match + RASGEF1B manual inclusion)
{'KANSL1', 'BAALC', 'TENM2', 'SPPL2A', 'NEGR1', 'SLC6A1', 'FAM171A1', 'MPDZ', 'IRAK1BP1', 'QKI', 'CABLES1', 'NRP1', 'NBAS', 'RERG', 'ARL17B', 'LINGO1', 'LRIG1', 'FBXL7', 'FAT1', 'GRM3', 'SORT1', 'CTNND2', 'LRRC4C', 'ABCA1', 'PRKG1', 'NPAS2', 'NRXN1'}
Mic: 68 genes overlap with GWAS (exact match + RASGEF1B manual inclusion)
{'KANSL1', 'ARHGAP15', 'INPP5D', 'AFF1', 'SMARCA2', 'MAP4K4', 'CTNNA1', 'ATP8B4', 'GAB2', 'ATM', 'PDE8A', 'ERC2', 'DENND3', 'SNX9', 'ELMO1', 'IQGAP2', 'QKI', 'SREBF2', 'EXOC4', 'FOXP2', 'APBA1', 'SDK1', 'PLCG2', 'TANC2', 'DST', 'HDAC9', 'SPG11', 'ST6GAL1', 'CELF2', 'MTMR3', 'RASGEF1C', 'ARL17B', 'PDE4B', 'DSCAM', 'NFAT5', 'TMEM163', 'LINGO1', 'DENND4A

In [5]:
# reran updated for fdr correction
import os
import joblib
from collections import defaultdict
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
import xlsxwriter

# --- BH helper ---
def bh_fdr(pvals):
    """Benjamini–Hochberg FDR for a list-like of p-values. Returns q-values in original order."""
    p = np.asarray(pvals, dtype=float)
    n = p.size
    if n == 0:
        return p
    order = np.argsort(p)
    ranks = np.arange(1, n + 1)
    p_sorted = p[order]
    q_sorted = p_sorted * n / ranks
    # enforce monotonicity from the end
    q_sorted = np.minimum.accumulate(q_sorted[::-1])[::-1]
    q = np.empty_like(q_sorted)
    q[order.argsort()] = np.minimum(q_sorted, 1.0)
    return q

# === Paths ===
base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
gwas_path = '/n/scratch/users/a/adm808/MONDO_0004975_associations_export.tsv'
background_path = '/home/adm808/NormalizedCellMatrixSyn18485175.parquet'
output_excel_path = '/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/GWAS_AD_overlap_summary.xlsx'

# === Cell type labels ===
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligodendrocyte Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}
cell_types = list(ct_labels.keys())

# === Dictionary to hold final predictors ===
predictive_genes_dict = {}

for cell_type in cell_types:
    print(f"Processing: {cell_type}")
    gene_presence = defaultdict(int)

    for split in range(1, 6):
        joblib_path = os.path.join(base_dir, cell_type, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(joblib_path):
            continue

        model = joblib.load(joblib_path)
        feature_names = model.feature_names_in_
        importances = model.feature_importances_

        for gene, imp in zip(feature_names, importances):
            if imp > 0:
                gene_presence[gene] += 1

    # Keep genes that are non-zero in ≥ 2 splits
    final_predictors = [gene for gene, count in gene_presence.items() if count >= 2]
    predictive_genes_dict[cell_type] = final_predictors
    print(f"{cell_type}: {len(final_predictors)} predictive genes")

# === Load GWAS genes ===
gwas_df = pd.read_csv(gwas_path, sep='\t', low_memory=False)
gwas_genes = set()
for genes in gwas_df['mappedGenes'].dropna():
    for g in genes.split(','):
        gwas_genes.add(g.strip().upper())

# === Load background genes ===
gene_matrix = pd.read_parquet(background_path)
background_genes = set(gene_matrix.index.str.upper())
print("Example background genes:", list(background_genes)[:10])

# === Setup Fisher results table ===
fisher_summary = []

# Add -log10 p-value column
gwas_df['pValue_num'] = pd.to_numeric(gwas_df['pValue'], errors='coerce')
gwas_df['MinusLog10P'] = -np.log10(gwas_df['pValue_num'])

# === Prepare Excel writer ===
with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:

    # --- Add title page first ---
    title_text = (
        "This workbook summarizes overlaps between predictive genes from our Alzheimer's classifiers "
        "and GWAS genes from the NHGRI-EBI catalog. Each sheet contains GWAS entries overlapping "
        "with predictive genes per cell type, including -log10(p) values. The 'Summary_Fisher_Test' "
        "sheet summarizes the Fisher exact test results per cell type. "
        "Q Value column shows Benjamini–Hochberg FDR across all 6 cell types."
    )
    title_df = pd.DataFrame({"Description": [title_text]})
    title_df.to_excel(writer, sheet_name="Title_Page", index=False)

    # === Then add cell-type sheets and summary ===
    for cell_type, pred_genes in predictive_genes_dict.items():
        pred_genes_upper = set(g.upper() for g in pred_genes)

        # Manual addition of RASGEF1B
        if "RASGEF1B" in pred_genes_upper and "RASGEF1B" in gwas_genes:
            overlap_genes = (pred_genes_upper & gwas_genes) | {"RASGEF1B"}
        else:
            overlap_genes = pred_genes_upper & gwas_genes

        # Fisher test
        a = len(overlap_genes)
        b = len(pred_genes_upper - overlap_genes)
        c = len(gwas_genes - overlap_genes)
        d = len(background_genes - (pred_genes_upper | gwas_genes))

        table = [[a, b], [c, d]]
        oddsratio, p_value = fisher_exact(table, alternative='two-sided')

        fisher_summary.append({
            "Cell Type": ct_labels[cell_type],
            "Predictive Genes": len(pred_genes_upper),
            "GWAS Genes": len(gwas_genes),
            "Overlap Genes": a,
            "Odds Ratio": oddsratio,
            "P Value": p_value,
        })

        # === Create filtered GWAS dataframe for this cell type ===
        overlap_genes_sorted = sorted(overlap_genes)
        gwas_df_filtered = gwas_df[gwas_df['mappedGenes'].apply(
            lambda x: any(gene in overlap_genes_sorted for gene in str(x).upper().split(',')) if pd.notna(x) else False
        )]

        # Save sheet per cell type
        gwas_df_filtered.drop(columns=["pValue_num"], inplace=True, errors='ignore')
        gwas_df_filtered.to_excel(writer, sheet_name=ct_labels[cell_type][:31], index=False)

    # === Create summary sheet ===
    fisher_df = pd.DataFrame(fisher_summary)
    fisher_df.sort_values("Cell Type", inplace=True)

    # Apply BH-FDR across the 6 cell types
    fisher_df["Q Value"] = bh_fdr(fisher_df["P Value"].values)

    fisher_df.to_excel(writer, sheet_name="Summary_Fisher_Test", index=False)

print(f"Workbook saved to: {output_excel_path}")

Processing: Ast
Ast: 175 predictive genes
Processing: Mic
Mic: 466 predictive genes
Processing: In
In: 108 predictive genes
Processing: Oli
Oli: 621 predictive genes
Processing: Opc
Opc: 835 predictive genes
Processing: Ex
Ex: 162 predictive genes
Example background genes: ['SIKE1', 'PPFIA2', 'GDPD1', 'RHBDD1', 'SPATA16', 'TUSC1', 'SERPINB2', 'CFAP36', 'COL6A3', 'NUDC']


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipykernel_3357086/2450735140.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gwas_df_filtered.drop(columns=["pValue_num"], inplace=True, errors='ignore')
/tmp/ipykernel_3357086/2450735140.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gwas_df_filtered.drop(columns=["pValue_num"], inplace=True, errors='ignore')
/tmp/ipykernel_3357086/2450735140.py:137: SettingWithCopyWarning: 
A value is trying to be set on a c

Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/GWAS_AD_overlap_summary.xlsx


/tmp/ipykernel_3357086/2450735140.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gwas_df_filtered.drop(columns=["pValue_num"], inplace=True, errors='ignore')


In [6]:
# reran
import os
import joblib
from collections import defaultdict
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact
import xlsxwriter

# --- BH helper ---
def bh_fdr(pvals):
    """Benjamini–Hochberg FDR for a list-like of p-values. Returns q-values in original order."""
    p = np.asarray(pvals, dtype=float)
    n = p.size
    if n == 0:
        return p
    order = np.argsort(p)
    ranks = np.arange(1, n + 1)
    p_sorted = p[order]
    q_sorted = p_sorted * n / ranks
    # enforce monotonicity from the end
    q_sorted = np.minimum.accumulate(q_sorted[::-1])[::-1]
    q = np.empty_like(q_sorted)
    q[order.argsort()] = np.minimum(q_sorted, 1.0)
    return q

# === Paths ===
base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_cerad"
gwas_path = '/n/scratch/users/a/adm808/MONDO_0004975_associations_export.tsv'
background_path = '/home/adm808/NormalizedCellMatrixSyn18485175.parquet'
output_excel_path = '/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/GWAS_CERAD_overlap_summary.xlsx'

# === Cell type labels ===
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligodendrocyte Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}
cell_types = list(ct_labels.keys())

# === Dictionary to hold final predictors ===
predictive_genes_dict = {}

for cell_type in cell_types:
    print(f"Processing: {cell_type}")
    gene_presence = defaultdict(int)

    for split in range(1, 6):
        joblib_path = os.path.join(base_dir, cell_type, f"split_{split}", "maximal_classifier.joblib")
        if not os.path.exists(joblib_path):
            continue

        model = joblib.load(joblib_path)
        feature_names = model.feature_names_in_
        importances = model.feature_importances_

        for gene, imp in zip(feature_names, importances):
            if imp > 0:
                gene_presence[gene] += 1

    # Keep genes that are non-zero in ≥ 2 splits
    final_predictors = [gene for gene, count in gene_presence.items() if count >= 2]
    predictive_genes_dict[cell_type] = final_predictors
    print(f"{cell_type}: {len(final_predictors)} predictive genes")

# === Load GWAS genes ===
gwas_df = pd.read_csv(gwas_path, sep='\t', low_memory=False)
gwas_genes = set()
for genes in gwas_df['mappedGenes'].dropna():
    for g in genes.split(','):
        gwas_genes.add(g.strip().upper())

# === Load background genes ===
gene_matrix = pd.read_parquet(background_path)
background_genes = set(gene_matrix.index.str.upper())
print("Example background genes:", list(background_genes)[:10])

# === Setup Fisher results table ===
fisher_summary = []

# Add -log10 p-value column
gwas_df['pValue_num'] = pd.to_numeric(gwas_df['pValue'], errors='coerce')
gwas_df['MinusLog10P'] = -np.log10(gwas_df['pValue_num'])

# === Prepare Excel writer ===
with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:

    # --- Add title page first ---
    title_text = (
        "This workbook summarizes overlaps between predictive genes from our CERAD pathology classifiers "
        "and GWAS genes from the NHGRI-EBI catalog. Each sheet contains GWAS entries overlapping "
        "with predictive genes per cell type, including -log10(p) values. The 'Summary_Fisher_Test' "
        "sheet summarizes the Fisher exact test results per cell type. Q Value is BH-FDR across all 6 cell types."
    )
    pd.DataFrame({"Description": [title_text]}).to_excel(writer, sheet_name="Title_Page", index=False)

    # === Then add cell-type sheets and summary ===
    for cell_type, pred_genes in predictive_genes_dict.items():
        pred_genes_upper = set(g.upper() for g in pred_genes)

        # Manual addition of RASGEF1B
        if "RASGEF1B" in pred_genes_upper and "RASGEF1B" in gwas_genes:
            overlap_genes = (pred_genes_upper & gwas_genes) | {"RASGEF1B"}
        else:
            overlap_genes = pred_genes_upper & gwas_genes

        # Fisher test
        a = len(overlap_genes)
        b = len(pred_genes_upper - overlap_genes)
        c = len(gwas_genes - overlap_genes)
        d = len(background_genes - (pred_genes_upper | gwas_genes))

        table = [[a, b], [c, d]]
        oddsratio, p_value = fisher_exact(table, alternative='two-sided')

        fisher_summary.append({
            "Cell Type": ct_labels[cell_type],
            "Predictive Genes": len(pred_genes_upper),
            "GWAS Genes": len(gwas_genes),
            "Overlap Genes": a,
            "Odds Ratio": oddsratio,
            "P Value": p_value,
        })

        # === Create filtered GWAS dataframe for this cell type ===
        overlap_genes_sorted = sorted(overlap_genes)
        gwas_df_filtered = gwas_df[gwas_df['mappedGenes'].apply(
            lambda x: any(gene in overlap_genes_sorted for gene in str(x).upper().split(',')) if pd.notna(x) else False
        )]

        # Save sheet per cell type
        gwas_df_filtered.drop(columns=["pValue_num"], inplace=True, errors='ignore')
        gwas_df_filtered.to_excel(writer, sheet_name=ct_labels[cell_type][:31], index=False)

    # === Create summary sheet ===
    fisher_df = pd.DataFrame(fisher_summary)
    fisher_df.sort_values("Cell Type", inplace=True)

    # Add BH-FDR Q Value across all 6 cell types
    fisher_df["Q Value"] = bh_fdr(fisher_df["P Value"].values)

    fisher_df.to_excel(writer, sheet_name="Summary_Fisher_Test", index=False)

print(f"Workbook saved to: {output_excel_path}")

Processing: Ast


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 w

Ast: 104 predictive genes
Processing: Mic


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.5.1 wh

Mic: 422 predictive genes
Processing: In


...

/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.5.1 wh

Ex: 18 predictive genes
Example background genes: ['SIKE1', 'PPFIA2', 'GDPD1', 'RHBDD1', 'SPATA16', 'TUSC1', 'SERPINB2', 'CFAP36', 'COL6A3', 'NUDC']


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log10
  result = getattr(ufunc, method)(*inputs, **kwargs)
/tmp/ipykernel_3357086/4103115956.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gwas_df_filtered.drop(columns=["pValue_num"], inplace=True, errors='ignore')
/tmp/ipykernel_3357086/4103115956.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gwas_df_filtered.drop(columns=["pValue_num"], inplace=True, errors='ignore')


Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/GWAS_CERAD_overlap_summary.xlsx


/tmp/ipykernel_3357086/4103115956.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gwas_df_filtered.drop(columns=["pValue_num"], inplace=True, errors='ignore')
/tmp/ipykernel_3357086/4103115956.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gwas_df_filtered.drop(columns=["pValue_num"], inplace=True, errors='ignore')
/tmp/ipykernel_3357086/4103115956.py:135: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  gwas_df_filtered.drop

In [7]:
import os
import joblib
import pandas as pd
import numpy as np
from collections import defaultdict
from scipy.stats import fisher_exact
from statsmodels.stats.contingency_tables import Table2x2
# reran with fdr correction

# --- BH helper ---
def bh_fdr(pvals):
    """Benjamini–Hochberg FDR correction."""
    p = np.asarray(pvals, dtype=float)
    n = p.size
    if n == 0:
        return p
    order = np.argsort(p)
    ranks = np.arange(1, n + 1)
    p_sorted = p[order]
    q_sorted = p_sorted * n / ranks
    q_sorted = np.minimum.accumulate(q_sorted[::-1])[::-1]  # enforce monotonicity
    q = np.empty_like(q_sorted)
    q[order.argsort()] = np.minimum(q_sorted, 1.0)
    return q

# === Background genes ===
gene_matrix = pd.read_parquet('/home/adm808/NormalizedCellMatrixSyn18485175.parquet')
background_genes = set(gene_matrix.index.str.upper())
print(f"Total background genes: {len(background_genes)}")

# === Cell type labels ===
ct_labels = {
    'Ast': 'Astrocytes',
    'Mic': 'Microglia',
    'In': 'Inhibitory Neurons',
    'Oli': 'Oligodendrocytes',
    'Opc': 'Oligodendrocyte Progenitor Cells',
    'Ex': 'Excitatory Neurons'
}
cell_types = list(ct_labels.keys())

# === Paths ===
ad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_new"
cerad_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Multirun_cell_on_cell_genes_cerad"
de_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/Differential_Expression_Final/Fixed/"
cerad_de_base_dir = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Final_Outputs_Figures/Differential_Expression_Final/CERAD"

de_files = {
    "Ast": "poisson_DE_results_Ast.csv",
    "Mic": "poisson_DE_results_Mic.csv",
    "In": "poisson_DE_results_In.csv",
    "Oli": "poisson_DE_results_Oli.csv",
    "Opc": "poisson_DE_results_Opc.csv",
    "Ex": "poisson_DE_results_Ex.csv"
}

# === Collect results for all cell types/tests ===
all_results = []

# === Prepare Excel writer ===
output_excel_path = "/n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/All_NonGWAS_Fisher_Tests.xlsx"
with pd.ExcelWriter(output_excel_path, engine='xlsxwriter') as writer:

    # --- Title page ---
    title_text = (
        "This workbook summarizes all non-GWAS Fisher tests run per cell type.\n\n"
        "Included tests:\n"
        "1. AD predictors vs CERAD predictors\n"
        "2. AD predictors vs AD DE genes\n"
        "3. CERAD predictors vs CERAD DE genes\n"
        "4. AD DE genes vs CERAD DE genes\n\n"
        "QValue is Benjamini–Hochberg FDR, calculated separately for each test type across all 6 cell types."
    )
    pd.DataFrame({"Description": [title_text]}).to_excel(writer, sheet_name="Title_Page", index=False)

    for cell_short in cell_types:
        cell_name = ct_labels[cell_short]
        rows = []

        # === Get AD predictors ===
        gene_presence_ad = defaultdict(int)
        for split in range(1, 6):
            path = os.path.join(ad_base_dir, cell_short, f"split_{split}", "maximal_classifier.joblib")
            if os.path.exists(path):
                model = joblib.load(path)
                for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
                    if imp > 0:
                        gene_presence_ad[gene.upper()] += 1
        ad_predictors = {g for g, cnt in gene_presence_ad.items() if cnt >= 2}

        # === Get CERAD predictors ===
        gene_presence_cerad = defaultdict(int)
        for split in range(1, 6):
            path = os.path.join(cerad_base_dir, cell_short, f"split_{split}", "maximal_classifier.joblib")
            if os.path.exists(path):
                model = joblib.load(path)
                for gene, imp in zip(model.feature_names_in_, model.feature_importances_):
                    if imp > 0:
                        gene_presence_cerad[gene.upper()] += 1
        cerad_predictors = {g for g, cnt in gene_presence_cerad.items() if cnt >= 2}

        # === Get AD DE genes ===
        # ad_de_df = pd.read_csv(os.path.join(de_base_dir, de_files[cell_short]))
        # ad_de_df["gene_upper"] = ad_de_df["gene"].str.upper()
        # ad_de_genes = set(ad_de_df[ad_de_df["p_adj"] < 0.05]["gene_upper"])

        # # === Get CERAD DE genes ===
        # cerad_de_df = pd.read_csv(os.path.join(cerad_de_base_dir, de_files[cell_short]))
        # cerad_de_df["gene_upper"] = cerad_de_df["gene"].str.upper()
        # cerad_de_genes = set(cerad_de_df[cerad_de_df["p_adj"] < 0.05]["gene_upper"])

                # === Get AD DE genes ===
        ad_de_df = pd.read_csv(os.path.join(de_base_dir, de_files[cell_short]))
        ad_de_df["gene_upper"] = ad_de_df["gene"].str.upper()
        ad_de_genes = set(
            ad_de_df[(ad_de_df["p_adj"] < 0.05) & (ad_de_df["log2FC"].abs() > 0.25)]["gene_upper"]
        )

        # === Get CERAD DE genes ===
        cerad_de_df = pd.read_csv(os.path.join(cerad_de_base_dir, de_files[cell_short]))
        cerad_de_df["gene_upper"] = cerad_de_df["gene"].str.upper()
        cerad_de_genes = set(
            cerad_de_df[(cerad_de_df["p_adj"] < 0.05) & (cerad_de_df["log2FC"].abs() > 0.25)]["gene_upper"]
        )

        # --- Define helper for adding results ---
        def add_test(test_name, set1, set2):
            overlap = set1 & set2
            a = len(overlap)
            b = len(set1 - overlap)
            c = len(set2 - overlap)
            d = len(background_genes - (set1 | set2))
            table = [[a, b], [c, d]]
            or_val, p_val = fisher_exact(table, alternative='greater')
            ci_low, ci_high = Table2x2(table).oddsratio_confint()
            row = {
                "CellType": cell_name,
                "Test": test_name,
                "Num_Set1_Genes": len(set1),
                "Num_Set2_Genes": len(set2),
                "Overlap": a,
                "OddsRatio": or_val,
                "CI_Low": ci_low,
                "CI_High": ci_high,
                "PValue": p_val,
                "Overlap_Genes": ", ".join(sorted(overlap))
            }
            rows.append(row)
            all_results.append(row)

        # --- Run the 4 tests ---
        add_test("AD predictors vs CERAD predictors", ad_predictors, cerad_predictors)
        add_test("AD predictors vs AD DE genes", ad_predictors, ad_de_genes)
        add_test("CERAD predictors vs CERAD DE genes", cerad_predictors, cerad_de_genes)
        add_test("AD DE genes vs CERAD DE genes", ad_de_genes, cerad_de_genes)

        # Save sheet (QValues will be added later after FDR)
        pd.DataFrame(rows).to_excel(writer, sheet_name=cell_name[:31], index=False)

    # === FDR correction per test type ===
    all_df = pd.DataFrame(all_results)
    all_df["QValue"] = all_df.groupby("Test")["PValue"].transform(lambda p: bh_fdr(p.values))

    # Write updated per-cell sheets with QValue
    for cell_name in all_df["CellType"].unique():
        df_cell = all_df[all_df["CellType"] == cell_name].copy()
        df_cell.to_excel(writer, sheet_name=cell_name[:31], index=False)

    # Optional: summary sheet with all results
    all_df.to_excel(writer, sheet_name="All_Tests_Summary", index=False)

print(f"Workbook saved to: {output_excel_path}")

Total background genes: 17926


/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SimpleImputer from version 1.5.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/n/groups/patel/adithya/scenv/lib64/python3.9/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ColumnTransformer from version 1.5.1 w

Workbook saved to: /n/groups/patel/adithya/Alz_Outputs/Final_Outputs/Data/All_NonGWAS_Fisher_Tests.xlsx
